# CampusGuard fight model training
Run the cells top to bottom. Before you start: **Runtime > Change runtime type > T4 GPU**.

Put your videos in Google Drive like this:
```
MyDrive/campusguard_data/fight/      fights (staged is fine)
MyDrive/campusguard_data/nofight/    classrooms, corridors, playground, canteen, sport, play-fighting
MyDrive/campusguard_data/segments.csv   (optional) video,start_s,end_s
```
Only use footage you are allowed to use (your own staged clips, or the school's written permission).

In [ ]:
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - change the runtime type')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!wget -q -O fightkit.py https://raw.githubusercontent.com/hisir-sys/campusguard/main/training/fightkit.py
!ls /content/drive/MyDrive/campusguard_data

## 1. Prepare (decodes every video once; a few minutes)

In [ ]:
!python fightkit.py prepare --data /content/drive/MyDrive/campusguard_data --cache /content/cache

## 2. Train (watch false@.85 - it should be tiny - and recall@.85 - it should be high)

In [ ]:
!python fightkit.py train --cache /content/cache --out /content/drive/MyDrive/campusguard_data/fight_model.pth --epochs 8 --workers 2

## 3. Check on calm footage that was NOT used for training
Put 30+ minutes of fight-free footage from the real school in `check/`. The numbers are false alarms per hour per camera.

In [ ]:
import glob
vids = glob.glob('/content/drive/MyDrive/campusguard_data/check/*.mp4')
!python fightkit.py check --model /content/drive/MyDrive/campusguard_data/fight_model.pth --videos {' '.join(vids)}

The trained model is `fight_model.pth` in your Drive folder. Download it, and send it back.